# Three-Judge Comparison

Compare three LLM-as-a-judge evaluation runs over the same dataset.
This notebook aligns samples across all judges, compares mean metrics,
shows pairwise agreement, and highlights where judges disagree most.

In [ ]:
import itertools
import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.tracking import MlflowClient

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

In [ ]:
# Config
MLFLOW_TRACKING_URI = "http://localhost:8567"

# Internal labels; human-readable labels are derived from run params.
RUNS = {
    "judge_a": "53fdfeda71b644028d483906ed9cb406",
    "judge_b": "c8da3550a0a145f1aac8805d2ee6c733",
    "judge_c": "38c7e3a1d5b14453a2cdb78089fa7df8",
}

RAGAS_TABLE_ARTIFACT = "ragas_scores.json"
METRIC_COLS = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "answer_relevance",
    "factual_correctness",
    "factual_correctness_recall",
]
JOIN_KEYS = ["user_input", "subdomain", "question_class"]

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

In [ ]:
def list_artifacts_recursive(run_id: str, path: str = ""):
    items = []
    for art in client.list_artifacts(run_id, path):
        items.append(art.path)
        if art.is_dir:
            items.extend(list_artifacts_recursive(run_id, art.path))
    return items


def normalize_logged_table(json_obj) -> pd.DataFrame:
    if isinstance(json_obj, list):
        return pd.DataFrame(json_obj)
    if isinstance(json_obj, dict):
        if "data" in json_obj and "columns" in json_obj:
            return pd.DataFrame(json_obj["data"], columns=json_obj["columns"])
        return pd.DataFrame(json_obj)
    raise ValueError("Unsupported ragas_scores.json structure")


def load_ragas_table_for_run(run_id: str) -> pd.DataFrame:
    matches = [p for p in list_artifacts_recursive(run_id) if p.endswith(RAGAS_TABLE_ARTIFACT)]
    if not matches:
        raise FileNotFoundError(f"No {RAGAS_TABLE_ARTIFACT} in run {run_id}")
    local_path = client.download_artifacts(run_id, matches[0])
    with open(local_path, "r", encoding="utf-8") as f:
        payload = json.load(f)
    df = normalize_logged_table(payload)
    for col in METRIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def judge_label(info, fallback):
    p = info["params"]
    model = p.get("judge_model") or p.get("judge_llm_model")
    provider = p.get("judge_provider")
    if model and provider:
        return f"{provider}/{model}"
    return model or fallback

## 1. Fetch run metadata and logged means

In [ ]:
runs_info = {}
for label, run_id in RUNS.items():
    run = client.get_run(run_id)
    runs_info[label] = {
        "run_id": run_id,
        "params": dict(run.data.params),
        "metrics": dict(run.data.metrics),
        "start_time": pd.to_datetime(run.info.start_time, unit="ms"),
        "status": run.info.status,
    }

JUDGE_LABELS = {label: judge_label(info, label) for label, info in runs_info.items()}
JUDGE_ORDER = list(RUNS.keys())

params_df = pd.DataFrame({JUDGE_LABELS[label]: info["params"] for label, info in runs_info.items()})
display(params_df)

metrics_df = pd.DataFrame({JUDGE_LABELS[label]: info["metrics"] for label, info in runs_info.items()})
metric_order = [m for m in METRIC_COLS if m in metrics_df.index] + [m for m in metrics_df.index if m not in METRIC_COLS]
metrics_df = metrics_df.loc[metric_order]

for left, right in itertools.combinations(JUDGE_ORDER, 2):
    col = f"diff ({JUDGE_LABELS[right]} - {JUDGE_LABELS[left]})"
    metrics_df[col] = metrics_df[JUDGE_LABELS[right]] - metrics_df[JUDGE_LABELS[left]]

metrics_df.round(4)

## 2. Load and align per-sample scores

In [ ]:
per_run_tables = {}
for label, run_id in RUNS.items():
    df = load_ragas_table_for_run(run_id)
    df["run_label"] = label
    df["judge"] = JUDGE_LABELS[label]
    per_run_tables[label] = df
    print(f"{label} ({JUDGE_LABELS[label]}): {len(df)} rows")

metric_cols_present = [c for c in METRIC_COLS if c in next(iter(per_run_tables.values())).columns]

sample_sets = {
    label: set(map(tuple, df[JOIN_KEYS].fillna("").astype(str).itertuples(index=False, name=None)))
    for label, df in per_run_tables.items()
}
shared_samples = set.intersection(*sample_sets.values())
all_samples = set.union(*sample_sets.values())
print(f"Shared across all judges: {len(shared_samples)}")
print(f"In at least one judge: {len(all_samples)}")
for label in JUDGE_ORDER:
    print(f"Only in {label}: {len(sample_sets[label] - shared_samples)}")

wide = None
for label in JUDGE_ORDER:
    sub = per_run_tables[label][JOIN_KEYS + metric_cols_present].copy()
    sub = sub.rename(columns={c: f"{c}__{label}" for c in metric_cols_present})
    wide = sub if wide is None else wide.merge(sub, on=JOIN_KEYS, how="inner")

for c in metric_cols_present:
    cols = [f"{c}__{label}" for label in JUDGE_ORDER]
    wide[f"{c}__range"] = wide[cols].max(axis=1) - wide[cols].min(axis=1)
    wide[f"{c}__std"] = wide[cols].std(axis=1)

print(f"Aligned rows (inner join across all judges): {len(wide)}")
wide.head()

## 3. Overall metric comparison and pairwise agreement

In [ ]:
overall_rows = []
agreement_rows = []

for c in metric_cols_present:
    row = {"metric": c}
    metric_values = []
    for label in JUDGE_ORDER:
        col = f"{c}__{label}"
        m = wide[col].mean()
        row[JUDGE_LABELS[label]] = m
        metric_values.append(m)

    pairwise_mae = []
    for left, right in itertools.combinations(JUDGE_ORDER, 2):
        left_col = f"{c}__{left}"
        right_col = f"{c}__{right}"
        a = wide[left_col]
        b = wide[right_col]
        mask = a.notna() & b.notna()
        a_, b_ = a[mask], b[mask]
        pairwise_mae.append((a_ - b_).abs().mean() if len(a_) else np.nan)
        agreement_rows.append(
            {
                "metric": c,
                "judge_left": JUDGE_LABELS[left],
                "judge_right": JUDGE_LABELS[right],
                "n": int(mask.sum()),
                "pearson": a_.corr(b_, method="pearson") if len(a_) > 1 else np.nan,
                "spearman": a_.corr(b_, method="spearman") if len(a_) > 1 else np.nan,
                "mae": (a_ - b_).abs().mean() if len(a_) else np.nan,
                "exact_agreement": np.isclose(a_, b_).mean() if len(a_) else np.nan,
            }
        )

    row["range_of_means"] = np.nanmax(metric_values) - np.nanmin(metric_values)
    row["mean_pairwise_mae"] = np.nanmean(pairwise_mae)
    row["mean_sample_range"] = wide[f"{c}__range"].mean()
    overall_rows.append(row)

overall = pd.DataFrame(overall_rows).set_index("metric").round(4)
agreement_df = pd.DataFrame(agreement_rows).round(4)

display(overall)
display(agreement_df)

In [ ]:
x = np.arange(len(metric_cols_present))
bar_w = 0.24
offsets = np.linspace(-(len(JUDGE_ORDER) - 1) / 2, (len(JUDGE_ORDER) - 1) / 2, len(JUDGE_ORDER)) * bar_w

fig, ax = plt.subplots(figsize=(11, 4.5))
for offset, label in zip(offsets, JUDGE_ORDER):
    ax.bar(x + offset, [overall.loc[c, JUDGE_LABELS[label]] for c in metric_cols_present], bar_w, label=JUDGE_LABELS[label])

ax.set_xticks(x)
ax.set_xticklabels(metric_cols_present, rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("mean score")
ax.set_title("Mean metric score per judge")
ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, int(np.ceil(len(metric_cols_present) / 2)), figsize=(12, 6))
axes = np.array(axes).ravel()
for i, c in enumerate(metric_cols_present):
    ax = axes[i]
    vals = wide[f"{c}__range"].dropna()
    ax.hist(vals, bins=20, color="steelblue", edgecolor="white")
    ax.axvline(vals.mean(), color="red", linestyle="--", linewidth=1, label=f"mean={vals.mean():.3f}")
    ax.set_title(c)
    ax.set_xlabel("per-sample range across judges")
    ax.legend(fontsize=8)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()

## 4. Largest disagreements and group-level spread

In [ ]:
TOP_N = 10
for c in metric_cols_present:
    judge_cols = [f"{c}__{label}" for label in JUDGE_ORDER]
    display_cols = JOIN_KEYS + judge_cols + [f"{c}__range", f"{c}__std"]
    sub = wide[display_cols].sort_values(f"{c}__range", ascending=False).head(TOP_N)
    if sub[f"{c}__range"].max() == 0:
        continue
    renamed = sub.rename(columns={f"{c}__{label}": JUDGE_LABELS[label] for label in JUDGE_ORDER})
    print(f"\n=== Top {TOP_N} disagreements: {c} ===")
    print(renamed.to_string(index=False, max_colwidth=80))


def group_compare(df_wide: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for group_val, sub in df_wide.groupby(group_col, dropna=False):
        for c in metric_cols_present:
            judge_means = [sub[f"{c}__{label}"].mean() for label in JUDGE_ORDER]
            row = {
                group_col: group_val,
                "metric": c,
                "n": len(sub),
                "range_of_means": np.nanmax(judge_means) - np.nanmin(judge_means),
            }
            for label in JUDGE_ORDER:
                row[JUDGE_LABELS[label]] = sub[f"{c}__{label}"].mean()
            rows.append(row)
    return pd.DataFrame(rows)


by_qclass = group_compare(wide, "question_class")
by_subdomain = group_compare(wide, "subdomain")

display(by_qclass.pivot(index="question_class", columns="metric", values="range_of_means").round(4))
display(by_subdomain.pivot(index="subdomain", columns="metric", values="range_of_means").round(4))

In [ ]:
# Uncomment to export tables.
# wide.to_csv("judge_3way_per_sample.csv", index=False)
# overall.to_csv("judge_3way_overall.csv")
# agreement_df.to_csv("judge_3way_pairwise_agreement.csv", index=False)
# by_qclass.to_csv("judge_3way_by_qclass.csv", index=False)
# by_subdomain.to_csv("judge_3way_by_subdomain.csv", index=False)
print("Done.")